# 🌏 NZ Earthquake Declustering — Method 4: Autoencoder (Deep Learning)
### Unsupervised anomaly detection + feature learning

**Core idea:** Train a neural network to *compress* then *reconstruct* each earthquake's features.
Background (non-crisis) events reconstruct well — they are 'normal'.
Crisis events are unusual patterns that reconstruct with **higher error**.

**Architecture:** Input(22) → Encoder → Latent(4) → Decoder → Output(22)

```
 Input    Encoder              Latent    Decoder              Output
  22  →  [64→32→16→8]  →   4D space  →  [8→16→32→64]  →   22
```

**Two uses in this notebook:**
1. **Reconstruction error** → anomaly score → non-crisis (low error) vs crisis (high error)
2. **Latent space** → 4D representation → cluster with GMM or HDBSCAN for richer declustering

---
### Why Autoencoder over KDM/GMM for NZ?
- Learns **non-linear** feature interactions (KDM only linear SOM mapping)
- No distributional assumptions (unlike GMM's Gaussian constraint)
- Latent space captures complex seismicity patterns specific to NZ tectonics
- Can generalise to new events at test time (real-time applicable)

## 📦 Step 0 — Install & Import
> Requires PyTorch. CPU-only is fine on your workstation for this dataset size.

In [ ]:
# !pip install torch scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'✅ PyTorch {torch.__version__} | Device: {DEVICE}')
except ImportError:
    raise ImportError('Run: pip install torch')

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
def out(f): return str(OUTPUT_DIR / f)

print('✅ All imports done!')

---
## 📂 Step 1 — Load & Clean Data

In [ ]:
YOUR_CSV = 'som_feature_nz_real_catalog.csv'

df = pd.read_csv(YOUR_CSV, low_memory=False)
t_cols = [f'T{i}' for i in range(1,11)]
r_cols = [f'R{i}' for i in range(1,11)]
required = t_cols + r_cols + ['Mn','bval']

df_clean = df.dropna(subset=required).copy()
if 'i+' in df_clean.columns:
    df_clean = df_clean[df_clean['i+'] > 0].copy()
df_clean = df_clean.reset_index(drop=True)
print(f'Clean shape: {df_clean.shape}')

---
## 🧮 Step 2 — Feature Matrix & Data Loader

In [ ]:
feature_cols = t_cols + r_cols + ['Mn','bval']

X = df_clean[feature_cols].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

print(f'Feature matrix : {X_scaled.shape}')
print(f'Memory         : {X_scaled.nbytes/1e6:.1f} MB')

# ── HYPERPARAMETERS ──────────────────────────────────────
BATCH_SIZE   = 2048
LATENT_DIM   = 4      # 4D latent space (tune: 2, 4, 8, 16)
LEARNING_RATE = 1e-3
N_EPOCHS     = 50     # increase to 100 for better training
# ─────────────────────────────────────────────────────────

tensor_X = torch.tensor(X_scaled)
dataset  = TensorDataset(tensor_X)
loader   = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=(DEVICE.type=='cuda'))

print(f'\nDataLoader: {len(loader)} batches of {BATCH_SIZE}')

---
## 🧠 Step 3 — Define the Autoencoder Architecture

```
Encoder: 22 → 64 → 32 → 16 → LATENT_DIM
Decoder: LATENT_DIM → 16 → 32 → 64 → 22
```

We use **BatchNorm** + **LeakyReLU** for stable training on seismic data.
The bottleneck forces the network to learn the most informative compressed representation.

In [ ]:
class SeismicAutoencoder(nn.Module):
    def __init__(self, input_dim=22, latent_dim=4):
        super().__init__()

        # ── Encoder ──────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),

            nn.Linear(16, latent_dim)
        )

        # ── Decoder ──────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),

            nn.Linear(16, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),

            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z    = self.encoder(x)
        x_re = self.decoder(z)
        return x_re, z

    def encode(self, x):
        return self.encoder(x)


INPUT_DIM = X_scaled.shape[1]   # 22
model     = SeismicAutoencoder(input_dim=INPUT_DIM, latent_dim=LATENT_DIM).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {n_params:,}')

---
## 🏋️ Step 4 — Train the Autoencoder

Loss = **Mean Squared Error** between input and reconstruction.
A lower reconstruction loss after training = the network learned the data's structure well.

**Expected training time on workstation:**
- 380k events, 50 epochs, batch=2048 → ~5–10 minutes (CPU) / ~2 min (GPU)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
criterion = nn.MSELoss()

train_losses = []
print(f'Training on {DEVICE} for {N_EPOCHS} epochs...')

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for (batch_x,) in loader:
        batch_x = batch_x.to(DEVICE)
        optimizer.zero_grad()
        recon, _ = model(batch_x)
        loss     = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    train_losses.append(avg_loss)
    scheduler.step()

    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}')

# Plot training curve
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(train_losses, 'b-', lw=1.5)
ax.set_title('Autoencoder Training Loss (MSE)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out('AE_training_loss.png'), dpi=150)
plt.show()
print(f'\n✅ Training complete. Final loss: {train_losses[-1]:.6f}')

# Save model
torch.save(model.state_dict(), out('NZ_autoencoder_weights.pt'))
print('✅ Model weights saved → NZ_autoencoder_weights.pt')

---
## 📐 Step 5 — Compute Reconstruction Errors

For each event, compute the **per-event MSE** between input and reconstruction.

```
Low  reconstruction error → event is 'normal' → NON-CRISIS (background)
High reconstruction error → event is 'unusual' → CRISIS (aftershock / swarm)
```

We threshold using the **90th percentile** of reconstruction errors.
Events above threshold are unusual = likely crisis events.

In [ ]:
model.eval()
all_errors  = []
all_latents = []

print('Computing reconstruction errors...')
with torch.no_grad():
    for i in range(0, len(X_scaled), BATCH_SIZE):
        batch = torch.tensor(X_scaled[i:i+BATCH_SIZE]).to(DEVICE)
        recon, latent = model(batch)
        # Per-event MSE
        err = ((recon - batch) ** 2).mean(dim=1).cpu().numpy()
        all_errors.append(err)
        all_latents.append(latent.cpu().numpy())

recon_errors = np.concatenate(all_errors)     # shape: (n_events,)
latent_space = np.concatenate(all_latents)    # shape: (n_events, LATENT_DIM)

print(f'Reconstruction errors — mean: {recon_errors.mean():.4f}  '
      f'std: {recon_errors.std():.4f}  '
      f'max: {recon_errors.max():.4f}')

# Plot error distribution
fig, axes = plt.subplots(1,2, figsize=(14,5))
axes[0].hist(recon_errors, bins=100, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Reconstruction Error Distribution', fontsize=12)
axes[0].set_xlabel('MSE per event'); axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

axes[1].hist(np.log1p(recon_errors), bins=100, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].set_title('Log Reconstruction Error', fontsize=12)
axes[1].set_xlabel('log(1 + MSE)'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(out('AE_reconstruction_errors.png'), dpi=150)
plt.show()

---
## 🏷️ Step 6 — Classify Events

**Two strategies — run both and compare:**

**Strategy A: Error threshold**
High reconstruction error → crisis. Threshold = configurable percentile.

**Strategy B: GMM on latent space**
Cluster the 4D latent representation — richer than just using the error scalar.
This is the most powerful approach: the autoencoder *learns* the best representation,
then GMM clusters it.

In [ ]:
# ── Strategy A: Reconstruction error threshold ────────────
THRESHOLD_PERCENTILE = 75   # top 25% errors = crisis (tune this)
threshold = np.percentile(recon_errors, THRESHOLD_PERCENTILE)
print(f'Error threshold ({THRESHOLD_PERCENTILE}th percentile): {threshold:.4f}')

# Convert error to probability (sigmoid-style normalisation)
log_errors   = np.log1p(recon_errors)
log_min, log_max = log_errors.min(), log_errors.max()
P_crisis_A   = (log_errors - log_min) / (log_max - log_min + 1e-10)
label_A      = np.where(recon_errors >= threshold, 'crisis', 'non_crisis')

n_c_A  = (label_A=='crisis').sum()
n_nc_A = (label_A=='non_crisis').sum()
print(f'Strategy A → Crisis: {n_c_A:,} ({100*n_c_A/len(label_A):.1f}%)  '
      f'Non-crisis: {n_nc_A:,} ({100*n_nc_A/len(label_A):.1f}%)')

# ── Strategy B: GMM on latent space ──────────────────────
print('\nFitting GMM on latent space...')
gmm_latent = GaussianMixture(n_components=2, covariance_type='full',
                              random_state=42, n_init=5)
gmm_latent.fit(latent_space)
proba_B      = gmm_latent.predict_proba(latent_space)
hard_B       = gmm_latent.predict(latent_space)

# Map component with smaller mean T to crisis
t_indices = [feature_cols.index(c) for c in t_cols]
comp_T = [X[hard_B==c][:, t_indices].mean() if (hard_B==c).sum()>0 else np.inf
          for c in range(2)]
crisis_comp_B = np.argmin(comp_T)
P_crisis_B    = proba_B[:, crisis_comp_B]
label_B       = np.where(P_crisis_B >= 0.5, 'crisis', 'non_crisis')

n_c_B  = (label_B=='crisis').sum()
n_nc_B = (label_B=='non_crisis').sum()
print(f'Strategy B → Crisis: {n_c_B:,} ({100*n_c_B/len(label_B):.1f}%)  '
      f'Non-crisis: {n_nc_B:,} ({100*n_nc_B/len(label_B):.1f}%)')

# Use Strategy B as primary (richer)
df_labelled = df_clean.copy()
df_labelled['recon_error']    = recon_errors
df_labelled['P_crisis_error'] = P_crisis_A     # from Strategy A
df_labelled['P_crisis']       = P_crisis_B     # from Strategy B (primary)
df_labelled['P_non_crisis']   = 1.0 - P_crisis_B
df_labelled['label']          = label_B
df_labelled['label_error']    = label_A         # Strategy A alternative
df_labelled['confidence']     = np.abs(0.5 - df_labelled['P_crisis'].clip(0,1)) / 0.5
for d in range(LATENT_DIM):
    df_labelled[f'latent_{d}'] = latent_space[:, d]

print(f'\n✅ Primary classification (Strategy B) stored in "label" column')

---
## 📈 Step 7 — Visualisations
### Plot A — Latent Space Coloured by Classification

In [ ]:
# Use PCA on latent space for 2D visualisation if LATENT_DIM > 2
if LATENT_DIM > 2:
    pca_lat = PCA(n_components=2)
    lat_2d  = pca_lat.fit_transform(latent_space)
    x_label, y_label = 'Latent PC-1', 'Latent PC-2'
else:
    lat_2d  = latent_space[:, :2]
    x_label, y_label = 'Latent dim 1', 'Latent dim 2'

fig, axes = plt.subplots(1,3, figsize=(20,6))
idx_plot  = np.random.choice(len(lat_2d), min(40000,len(lat_2d)), replace=False)

# Crisis / non-crisis
c_mask  = df_labelled['label'].values[idx_plot] == 'crisis'
nc_mask = ~c_mask
axes[0].scatter(lat_2d[idx_plot][nc_mask,0], lat_2d[idx_plot][nc_mask,1],
                c='steelblue', s=0.5, alpha=0.4, label='Non-crisis')
axes[0].scatter(lat_2d[idx_plot][c_mask,0],  lat_2d[idx_plot][c_mask,1],
                c='crimson',   s=0.5, alpha=0.4, label='Crisis')
axes[0].set_title('Latent Space — Crisis vs Non-Crisis', fontsize=11)
axes[0].set_xlabel(x_label); axes[0].set_ylabel(y_label)
axes[0].legend(markerscale=8)

# Reconstruction error
sc1 = axes[1].scatter(lat_2d[idx_plot,0], lat_2d[idx_plot,1],
                      c=np.log1p(recon_errors[idx_plot]),
                      cmap='YlOrRd', s=0.5, alpha=0.5)
plt.colorbar(sc1, ax=axes[1], label='log(reconstruction error)')
axes[1].set_title('Latent Space — Reconstruction Error', fontsize=11)
axes[1].set_xlabel(x_label); axes[1].set_ylabel(y_label)

# P(crisis) from GMM on latent
sc2 = axes[2].scatter(lat_2d[idx_plot,0], lat_2d[idx_plot,1],
                      c=P_crisis_B[idx_plot],
                      cmap='RdYlGn_r', s=0.5, alpha=0.5, vmin=0, vmax=1)
plt.colorbar(sc2, ax=axes[2], label='P(crisis)')
axes[2].set_title('Latent Space — P(crisis)', fontsize=11)
axes[2].set_xlabel(x_label); axes[2].set_ylabel(y_label)

plt.suptitle('Autoencoder Latent Space — NZ Seismicity', fontsize=14)
plt.tight_layout()
plt.savefig(out('AE_latent_space.png'), dpi=150)
plt.show()

### Plot B — Spatial Map, Cumulative Curves & Strategy Comparison

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(16,12))
time_col  = 'time' if 'time' in df_labelled.columns else 'Year'
crisis     = df_labelled[df_labelled['label']=='crisis']
non_crisis = df_labelled[df_labelled['label']=='non_crisis']

# Spatial
axes[0,0].scatter(non_crisis['longitude'], non_crisis['latitude'],
                  c='steelblue', s=0.5, alpha=0.3, label=f'Non-crisis ({len(non_crisis):,})')
axes[0,0].scatter(crisis['longitude'],     crisis['latitude'],
                  c='crimson',   s=0.5, alpha=0.3, label=f'Crisis ({len(crisis):,})')
axes[0,0].set_title('AE — Spatial Classification', fontsize=11)
axes[0,0].set_xlabel('Longitude'); axes[0,0].set_ylabel('Latitude')
axes[0,0].legend(markerscale=8)

# P(crisis) map
sc = axes[0,1].scatter(df_labelled['longitude'], df_labelled['latitude'],
                        c=df_labelled['P_crisis'], cmap='RdYlGn_r',
                        s=0.5, alpha=0.4, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[0,1], label='P(crisis)')
axes[0,1].set_title('AE — P(crisis) Map', fontsize=11)
axes[0,1].set_xlabel('Longitude'); axes[0,1].set_ylabel('Latitude')

# Cumulative
df_s = df_labelled.sort_values(time_col)
N    = len(df_s)
c_s  = df_s[df_s['label']=='crisis']
nc_s = df_s[df_s['label']=='non_crisis']
axes[1,0].plot(df_s[time_col],  np.arange(N)/N, 'k--', lw=1, alpha=0.5)
axes[1,0].plot(c_s[time_col],  np.arange(len(c_s))/N, 'r:', lw=1.5, label=f'Crisis ({len(c_s):,})')
axes[1,0].plot(nc_s[time_col], np.arange(len(nc_s))/N, 'b-', lw=1.5, label=f'Non-crisis ({len(nc_s):,})')
axes[1,0].set_title('Cumulative Curves — Autoencoder', fontsize=11)
axes[1,0].set_xlabel('Time (decimal years)'); axes[1,0].set_ylabel('Proportion')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# Strategy A vs B comparison
agree   = (label_A == label_B).sum()
disagree = (label_A != label_B).sum()
axes[1,1].bar(['Agree','Disagree'], [agree, disagree],
              color=['seagreen','tomato'], alpha=0.8, edgecolor='white')
axes[1,1].set_title('Strategy A vs B Agreement', fontsize=11)
axes[1,1].set_ylabel('Number of events')
for i, (v, n) in enumerate(zip([agree,disagree], [agree,disagree])):
    axes[1,1].text(i, v+100, f'{100*v/len(label_A):.1f}%', ha='center', fontsize=12)

plt.suptitle('Autoencoder Declustering Results — NZ Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(out('AE_results_overview.png'), dpi=150)
plt.show()

---
## 💾 Step 8 — Save Results

In [ ]:
save_cols = [c for c in ['event','DateTime','latitude','longitude','depth',
             'magnitude','time'] if c in df_labelled.columns]
save_cols += ['recon_error','P_crisis','P_non_crisis','P_crisis_error',
              'label','label_error','confidence']
save_cols += [f'latent_{d}' for d in range(LATENT_DIM)]

df_labelled[save_cols].to_csv(out('NZ_Autoencoder_declustered.csv'), index=False)

n_c  = (df_labelled['label']=='crisis').sum()
n_nc = (df_labelled['label']=='non_crisis').sum()
print('✅ Saved → NZ_Autoencoder_declustered.csv')
print(f'\n══ Autoencoder Final Summary ══════════════════')
print(f'  Total       : {len(df_labelled):,}')
print(f'  Crisis      : {n_c:,}  ({100*n_c/len(df_labelled):.1f}%)')
print(f'  Non-crisis  : {n_nc:,}  ({100*n_nc/len(df_labelled):.1f}%)')
print(f'  Latent dim  : {LATENT_DIM}')
print(f'  Final loss  : {train_losses[-1]:.6f}')
print(f'══════════════════════════════════════════════')